In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-pd-update-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '02_pricing_pd'
    
    # load output from concat sensitivity
    print('Loading output from sensitivity analysis concatenation...')
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df = pd.read_csv(str_uri)
    
    # get the top feature
    print('Getting feature that helps the model most once removed...')
    str_col = df['feature'].iloc[0]
    
    # import the features to drop
    print('Importing features to drop...')
    str_filename = 'df_feats_to_drop.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_drop = list(pd.read_csv(str_uri)['feature'])
    
    # append
    print(f'Appending {str_col} to {str_uri}...')
    list_cols_drop.append(str_col)
    
    # create df and upload to s3
    print(f'Creating data frame and uploading to {str_uri}...')
    df = pd.DataFrame({'feature': list_cols_drop})
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-update-feats

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  55.81kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 303d20c679a3
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d25abc48fcf6
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> bda31bbc4cc9
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> ff8d6f29f40b
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 15856a312706
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 7496f3ba038e
Removing intermediate container 7496f3ba038e
 ---> 8caa0d93cbdb
Successfully built 8caa0d93cbdb
Successfully tagged genxii-pd-update-feats:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-update-feats' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-update-feats]
7b49cff52d50: Preparing
98ef36a6a930: Preparing
6003a43fab84: Preparing
d630e2305053: Preparing
3bd433acfe84: Preparing
09b55d38856d: Preparing
e073f5919ae5: Preparing
b3b414f01759: Preparing
8308f08f35ba: Preparing
c8203e562a8c: Preparing
09b55d38856d: Waiting
e073f5919ae5: Waiting
b3b414f01759: Waiting
8308f08f35ba: Waiting
c8203e562a8c: Waiting
7b49cff52d50: Pushed
6003a43fab84: Pushed
d630e2305053: Pushed
e073f5919ae5: Pushed
b3b414f01759: Pushed
8308f08f35ba: Pushed
3bd433acfe84: Pushed
09b55d38856d: Pushed
c8203e562a8c: Pushed
98ef36a6a930: Pushed
latest: digest: sha256:9efad2e2c207db13c74a7e72a2b201d456a320f1230f952dd8ddab9277bcdede size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:11:54 GMT',
                                      'x-amzn-requestid': '00d31295-2d4b-45a2-a7f1-4679a5690ad1'},
                      'HTTPStatusCode': 204,
                      'RequestId': '00d31295-2d4b-45a2-a7f1-4679a5690ad1',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '9efad2e2c207db13c74a7e72a2b201d456a320f1230f952dd8ddab9277bcdede',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-update-feats',
 'FunctionName': 'genxii-pd-update-feats',
 'LastModified': '2024-08-20T16:11:54.827+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-pd-update-feats'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1198',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 20 Aug 2024 16:11:55 GMT',
                                      'x-amzn-requestid': '68c00d72-d874-4a05-9ecb-da8f85854b49'},
                      'HTTPStatusCode': 201,
                      'RequestId': '68c00d72-d874-4a

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)